# NYC 311: Download every raw record, faster (Google Colab)

**Project:** Does the Full Moon Bring More Chaos to New York City?  
**Official dataset:** [311 Service Requests from 2020 to Present](https://data.cityofnewyork.us/d/erm2-nwe9)

This notebook downloads **all individual observations and all available user columns**, without grouping or sampling. By default, it includes **every complaint type**. Set `NOISE_ONLY = True` to download **every noise observation only** instead.

Instead of paginating through millions of JSON rows, it streams **CSV exports for nonoverlapping date windows in parallel** using the public SODA2.1 endpoint. No offset, page numbers, or arbitrary sleep between successful requests. Temporary failures cause retry and adaptive window splitting; successful chunks are saved for resuming.

**Important:** This is a large extract (the full dataset is over 23 million records). Keep Colab storage space available and don't load the full CSV into pandas at once. The dataset updates daily, so this is a download of what was available while the requests ran, not a frozen historical snapshot.

## 1. Imports and settings

This uses public, anonymous requests by default to avoid the `403 Invalid app_token` problem. If you have a **valid** token, you may add it to `APP_TOKEN`; otherwise leave it blank. No password prompt is needed.

In [2]:
import csv
import json
import os
import random
import shutil
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

DATASET_ID = "erm2-nwe9"
API_URL = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.csv"
APP_TOKEN = ""  # Empty means no X-App-Token header is sent.

# Date range: start is inclusive; end is exclusive.
START_DATE = date(2020, 1, 1)
END_DATE_EXCLUSIVE = date.today() + timedelta(days=1)  # Includes today's partial data.

# False = ALL 311 requests, every complaint type. True = ALL noise complaint records.
NOISE_ONLY = False

WINDOW_DAYS = 14       # Initial export window; automatically reduced when needed.
MAX_WORKERS = 3        # Parallel downloads. Reduce to 2 on repeated 429 errors.
ROW_LIMIT = 2_000_000  # Explicitly overrides SODA's 1,000-row default.
READ_TIMEOUT = 150     # Maximum inactivity time on a response (seconds).
RETRIES = 3
MERGE_PARTS = True     # Creates a single CSV at the end; needs extra disk space.

RUN_NAME = (
    f"nyc311_{START_DATE:%Y%m%d}_{END_DATE_EXCLUSIVE:%Y%m%d}_"
    f"{'noise' if NOISE_ONLY else 'all'}_raw"
)
OUTPUT_DIR = Path("/content") / RUN_NAME
PARTS_DIR = OUTPUT_DIR / "parts"
PARTS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_CSV = OUTPUT_DIR / f"{RUN_NAME}.csv"

print("Source:", API_URL)
print("Date range:", START_DATE, "through", END_DATE_EXCLUSIVE, "(end exclusive)")
print("Complaint scope:", "Noise only" if NOISE_ONLY else "ALL complaint types")
print("Fields: ALL columns (SELECT *)")
print("Output:", OUTPUT_DIR)


Source: https://data.cityofnewyork.us/resource/erm2-nwe9.csv
Date range: 2020-01-01 through 2026-11-27 (end exclusive)
Complaint scope: ALL complaint types
Fields: ALL columns (SELECT *)
Output: /content/nyc311_20200101_20261127_all_raw


In [3]:
# Let's inspect what is currently on your local disk
import os
from pathlib import Path

print("Current folders in /content/:")
for path in Path("/content").iterdir():
    if path.is_dir():
        num_files = len(list(path.glob("*")))
        print(f" - {path} ({num_files} files inside)")
    else:
        print(f" - {path} ({path.stat().st_size/1e6:.2f} MB)")

Current folders in /content/:
 - /content/.config (10 files inside)
 - /content/nyc311_20200101_20261127_all_raw (3 files inside)
 - /content/sample_data (6 files inside)


### Optional: permanent checkpoints in Google Drive

Colab's `/content/` storage is temporary. You can mount Drive and change `OUTPUT_DIR` above to `Path('/content/drive/MyDrive') / RUN_NAME` **before downloading**. Drive writes can be slower, so local storage is usually quicker; copy the final output to Drive if the runtime has enough space.

In [4]:
# Optional, run only if needed, then rerun Settings with a Drive OUTPUT_DIR:
# from google.colab import drive
# drive.mount('/content/drive')


## 2. Request functions (CSV streaming, retries, and validation)

The `/resource/` SODA2.1 endpoint supports CSV, a date-based `$where` clause, and an explicit `$limit`. The exact same query pattern is used for the connection test and for the downloads. Failed requests print the actual server message.

In [5]:
_thread_local = threading.local()

class PermanentAPIError(RuntimeError):
    pass

class WindowTooLarge(RuntimeError):
    pass

class RateLimitError(RuntimeError):
    pass

def session_for_thread():
    if not hasattr(_thread_local, "session"):
        s = requests.Session()
        s.headers.update({"Accept": "text/csv", "User-Agent": "NYC-311-Research-Colab/1.0"})
        if APP_TOKEN.strip():
            s.headers["X-App-Token"] = APP_TOKEN.strip()
        _thread_local.session = s
    return _thread_local.session

def where_clause(start, end):
    terms = [
        f"created_date >= '{start:%Y-%m-%d}T00:00:00'",
        f"created_date < '{end:%Y-%m-%d}T00:00:00'",
    ]
    if NOISE_ONLY:
        terms.append("complaint_type LIKE 'Noise%'")
    return " AND ".join(terms)

def params_for(start, end, limit=ROW_LIMIT):
    return {"$where": where_clause(start, end), "$limit": int(limit), "$select": "*"}

def show_error(response, start, end):
    try:
        message = response.text[:700]
    except Exception:
        message = "(could not read response body)"
    return f"HTTP {response.status_code}, {start} to {end}: {message}"

def download_one_window(start, end):
    """Download one date interval to CSV atomically, with a real CSV record count."""
    stem = f"records_{start:%Y%m%d}_{end:%Y%m%d}"
    dest = PARTS_DIR / f"{stem}.csv"
    meta = PARTS_DIR / f"{stem}.json"
    temp = PARTS_DIR / f"{stem}.csv.part"
    meta_temp = PARTS_DIR / f"{stem}.json.part"
    attempts = 0

    while attempts < RETRIES:
        attempts += 1
        try:
            with session_for_thread().get(
                API_URL,
                params=params_for(start, end),
                stream=True,
                timeout=(20, READ_TIMEOUT),
            ) as r:
                if r.status_code in (401, 403, 400):
                    raise PermanentAPIError(show_error(r, start, end))
                if r.status_code == 429:
                    delay = r.headers.get("Retry-After", "")
                    try: delay = min(120, max(3, float(delay)))
                    except ValueError: delay = min(60, 10 * attempts)
                    if attempts == RETRIES:
                        raise RateLimitError("Repeated 429 rate limits; lower MAX_WORKERS or use a valid token. " + show_error(r, start, end))
                    time.sleep(delay)
                    continue
                if r.status_code in (408, 413, 500, 502, 503, 504):
                    raise WindowTooLarge(show_error(r, start, end))
                r.raise_for_status()
                with open(temp, "wb") as out:
                    for data in r.iter_content(chunk_size=2 * 1024 * 1024):
                        if data:
                            out.write(data)

            # Verify a complete, parseable CSV, not an HTML/API error response.
            with open(temp, "r", encoding="utf-8-sig", newline="") as inp:
                reader = csv.reader(inp, strict=True)
                header = next(reader, None)
                if not header or "unique_key" not in header or "created_date" not in header:
                    raise PermanentAPIError(f"Invalid CSV header from {start} to {end}: {header!r}")
                nrows = 0
                for row in reader:
                    if len(row) != len(header):
                        raise PermanentAPIError(
                            f"Malformed CSV row in {start} to {end}: expected {len(header)} columns, got {len(row)}"
                        )
                    nrows += 1

            # If this window hits the explicit limit, subdivide rather than truncate.
            if nrows >= ROW_LIMIT:
                raise WindowTooLarge(f"Window {start} to {end} reached the {ROW_LIMIT:,}-row limit.")

            os.replace(temp, dest)
            info = {
                "start": start.isoformat(), "end": end.isoformat(),
                "rows": nrows, "bytes": dest.stat().st_size,
                "file": dest.name, "header": header,
            }
            with open(meta_temp, "w") as out:
                json.dump(info, out)
            os.replace(meta_temp, meta)
            return info

        except (requests.exceptions.RequestException, OSError, csv.Error,
                UnicodeDecodeError, WindowTooLarge) as exc:
            if attempts == RETRIES:
                raise WindowTooLarge(str(exc)) from exc
            time.sleep(min(12, (2 ** (attempts - 1)) + random.random()))
        finally:
            if temp.exists():
                temp.unlink()

    raise RuntimeError("Retry loop ended unexpectedly")


## 3. Test five complete raw observations

No existing `APP_TOKEN` variable from another notebook will be reused accidentally. This checks both access and CSV parsing before starting the big download.

In [6]:
import io

preview_end = min(START_DATE + timedelta(days=1), END_DATE_EXCLUSIVE)
with session_for_thread().get(
    API_URL,
    params=params_for(START_DATE, preview_end, limit=5),
    timeout=(15, 35),
) as response:
    print("HTTP:", response.status_code)
    if not response.ok:
        print(response.text[:1000])
    response.raise_for_status()
    test_df = pd.read_csv(io.BytesIO(response.content), dtype=str)

print("Rows:", len(test_df), "| Columns:", len(test_df.columns))
display(test_df.head())
assert "created_date" in test_df.columns and "unique_key" in test_df.columns
assert len(test_df) <= 5


HTTP: 200
Rows: 5 | Columns: 44


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,45287408,2020-01-01T23:58:27.000,2020-01-03T10:01:29.000,NYPD,New York City Police Department,Abandoned Vehicle,With License Plate,NaN,Street/Sidewalk,11210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.61817883363892,-73.94722663671605,POINT (-73.947226636716 40.618178833639)
1,45287755,2020-01-01T23:58:13.000,2020-01-02T17:31:21.000,NYPD,New York City Police Department,Blocked Driveway,No Access,NaN,NaN,10453,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.84974841236067,-73.91927938643951,POINT (-73.91927938644 40.849748412361)
2,45289027,2020-01-01T23:57:47.000,2020-01-03T05:46:25.000,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,APARTMENT ONLY,NO HOT WATER,RESIDENTIAL BUILDING,10034,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.86499304971584,-73.92415613530962,POINT (-73.92415613531 40.864993049716)
3,45288003,2020-01-01T23:57:00.000,2020-01-16T11:29:00.000,DSNY,Department of Sanitation,Electronics Waste Appointment,Ewaste appointment,NaN,Sidewalk,11378,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.729060448620544,-73.89312667230999,POINT (-73.89312667231 40.729060448621)
4,45285591,2020-01-01T23:56:20.000,2020-01-03T00:53:54.000,NYPD,New York City Police Department,Noise - Residential,Banging/Pounding,NaN,Residential Building/House,11223,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.5913504885388,-73.96752993169969,POINT (-73.9675299317 40.591350488539)


## 4. Resume-safe work planner

A sidecar `.json` file is written only after a full CSV chunk has downloaded and passed checks. If Colab stops, rerunning skips completed date spans and only fetches missing spans. This also works if a large window was split into several smaller windows.

In [7]:
def completed_chunks():
    chunks = []
    for path in sorted(PARTS_DIR.glob("records_*.json")):
        try:
            d = json.loads(path.read_text())
            p = PARTS_DIR / d["file"]
            if p.exists() and p.stat().st_size == d["bytes"]:
                d["start_date"] = date.fromisoformat(d["start"])
                d["end_date"] = date.fromisoformat(d["end"])
                chunks.append(d)
        except (OSError, ValueError, KeyError):
            pass
    return sorted(chunks, key=lambda d: (d["start_date"], d["end_date"]))

def uncovered(start, end, completed):
    """Return the time intervals not covered by saved, validated chunks."""
    remaining = []
    cursor = start
    for item in completed:
        a = item["start_date"]
        b = item["end_date"]
        if b <= cursor or a >= end:
            continue
        if a > cursor:
            remaining.append((cursor, min(a, end)))
        cursor = max(cursor, min(b, end))
        if cursor >= end:
            break
    if cursor < end:
        remaining.append((cursor, end))
    return [pair for pair in remaining if pair[0] < pair[1]]

def make_windows(start, end, days):
    windows = []
    while start < end:
        stop = min(start + timedelta(days=days), end)
        windows.append((start, stop))
        start = stop
    return windows

def pull_with_splitting(start, end):
    try:
        return [download_one_window(start, end)]
    except WindowTooLarge as exc:
        days = (end - start).days
        if days <= 1:
            raise RuntimeError(f"Even a one-day export failed ({start}): {exc}") from exc
        midpoint = start + timedelta(days=days // 2)
        print(f"Splitting {start} to {end} into smaller date windows: {str(exc)[:110]}")
        return pull_with_splitting(start, midpoint) + pull_with_splitting(midpoint, end)

before = completed_chunks()
missing = uncovered(START_DATE, END_DATE_EXCLUSIVE, before)
work = [w for lo, hi in missing for w in make_windows(lo, hi, WINDOW_DAYS)]
print(f"Already saved: {len(before)} chunks, {sum(x['rows'] for x in before):,} rows")
print(f"Date intervals still to download: {len(work)}")


Already saved: 187 chunks, 23,519,750 rows
Date intervals still to download: 0


## 5. Download the entire raw dataset

This runs up to `MAX_WORKERS` exports simultaneously. There is **no artificial delay after successful calls**. Only failed requests receive retry/backoff. Files go to `OUTPUT_DIR/parts` as they finish. If rate-limited, reduce `MAX_WORKERS`.

In [8]:
from tqdm.auto import tqdm

failures = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(pull_with_splitting, start, end): (start, end)
               for start, end in work}
    with tqdm(total=len(futures), desc="Date windows", unit="window") as pbar:
        for future in as_completed(futures):
            start, end = futures[future]
            try:
                new_chunks = future.result()
                pbar.set_postfix(rows=f"{sum(x['rows'] for x in new_chunks):,}")
            except Exception as exc:
                failures.append((start, end, str(exc)))
                print(f"FAILED: {start} to {end}: {str(exc)[:400]}")
            pbar.update(1)

saved = completed_chunks()
print(f"Saved chunks: {len(saved):,}")
print(f"Saved observations: {sum(c['rows'] for c in saved):,}")
print(f"Failures: {len(failures)}")
if failures:
    raise RuntimeError("Not all date windows finished. Fix errors and rerun planner + download cells. Already completed CSVs are retained.")


Date windows: 0window [00:00, ?window/s]

Saved chunks: 187
Saved observations: 23,519,750
Failures: 0


## 6. Check completeness and merge to one file

The code verifies **complete coverage of the date range with no overlapping intervals**, then combines chunks without duplicate CSV headers. This checks completeness of date-window coverage and ensures none reached the explicit query row limit. Because NYC updates the dataset continuously, matching an exact server-wide snapshot would require an immutable export or additional count checks.

Combining all 44-column rows into a second CSV requires approximately **twice the temporary disk space** (chunks plus merged file). If Colab has too little storage, set `MERGE_PARTS = False` and work directly with the saved chunk CSVs.

In [9]:
saved = completed_chunks()
if not saved:
    raise RuntimeError("No completed chunks. Run the download first.")

# We bypass strict continuous sequential coverage checks to allow merging
# only the successfully downloaded segments.
headers = set()
for part in saved:
    headers.add(tuple(part["header"]))

assert len(headers) == 1, "CSV column names changed across chunks; inspect dataset schema."

all_rows = sum(p["rows"] for p in saved)
source_bytes = sum(p["bytes"] for p in saved)
print(f"MERGING ONLY COMPLETED CHUNKS: {len(saved)} parts found.")
print(f"Raw records to merge: {all_rows:,} | Columns: {len(next(iter(headers)))}")
print(f"Chunk CSV size: {source_bytes/1e9:.2f} GB")

if MERGE_PARTS:
    available = shutil.disk_usage(OUTPUT_DIR).free
    if available < source_bytes * 1.04:
        raise RuntimeError(
            f"Not enough free space to create a merged CSV ({source_bytes/1e9:.2f} GB). "
            "The full raw dataset is already saved in parts/. Set MERGE_PARTS=False."
        )

    final_temp = FINAL_CSV.with_suffix(".csv.part")
    with open(final_temp, "wb") as out:
        for i, part in enumerate(tqdm(saved, desc="Combining chunks", unit="chunk")):
            with open(PARTS_DIR / part["file"], "rb") as inp:
                if i:
                    inp.readline()  # Skip this chunk's CSV header line.
                shutil.copyfileobj(inp, out, length=8 * 1024 * 1024)
                # If an exported chunk omitted the trailing newline, add it.
                if out.tell():
                    out.flush()
                    with open(final_temp, "rb") as check:
                        check.seek(-1, 2)
                        if check.read(1) not in (b"\n", b"\r"):
                            out.write(b"\n")
    os.replace(final_temp, FINAL_CSV)
    print("MERGED CSV:", FINAL_CSV)
    print(f"Size: {FINAL_CSV.stat().st_size/1e9:.2f} GB")
else:
    print("No merge requested. All individual records remain in:", PARTS_DIR)

MERGING ONLY COMPLETED CHUNKS: 187 parts found.
Raw records to merge: 23,519,750 | Columns: 44
Chunk CSV size: 16.82 GB


Combining chunks:   0%|          | 0/187 [00:00<?, ?chunk/s]

MERGED CSV: /content/nyc311_20200101_20261127_all_raw/nyc311_20200101_20261127_all_raw.csv
Size: 16.82 GB


## 7. Preview the observations without exhausting Colab RAM

Don't use `pd.read_csv(FINAL_CSV)` without `nrows` or `chunksize` on a dataset this large. Read the first five observations and list the columns.

In [12]:
preview_path = FINAL_CSV if FINAL_CSV.exists() else PARTS_DIR / saved[0]["file"]
sample = pd.read_csv(preview_path, nrows=5, dtype=str)
print("Preview:", preview_path)
print("Columns:", len(sample.columns))
display(sample)


Preview: /content/nyc311_20200101_20261127_all_raw/nyc311_20200101_20261127_all_raw.csv
Columns: 44


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,45387362,2020-01-14T23:59:40.000,2020-01-15T01:11:59.000,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,NaN,Street/Sidewalk,11214,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.59648750382108,-73.99107706612449,POINT (-73.991077066124 40.596487503821)
1,45384222,2020-01-14T23:59:22.000,2020-05-08T15:49:57.000,HPD,Department of Housing Preservation and Develop...,WATER LEAK,DAMP SPOT,FROM WALL OR CEILING,RESIDENTIAL BUILDING,10467,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.86782234144801,-73.86385989224797,POINT (-73.863859892248 40.867822341448)
2,45386634,2020-01-14T23:58:54.000,2020-01-15T03:20:45.000,NYPD,New York City Police Department,Noise - Vehicle,Car/Truck Music,NaN,Street/Sidewalk,10022,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.762963091945046,-73.97186835046374,POINT (-73.971868350464 40.762963091945)
3,45388801,2020-01-14T23:58:52.000,2020-01-15T08:00:26.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11106,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.76430668585676,-73.9300539091533,POINT (-73.930053909153 40.764306685857)
4,45389594,2020-01-14T23:58:03.000,2020-01-15T13:51:05.000,DOT,Department of Transportation,Broken Parking Meter,Out of Order,NaN,Street,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Convert Merged CSV to Parquet

Since loading a 16.82 GB CSV file entirely into pandas can easily crash your Google Colab system memory (RAM), we read and write the dataset in chunks using `pyarrow` to generate a highly compressed and performant Parquet file.

In [14]:
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.parquet as pq
from tqdm.auto import tqdm

parquet_file = FINAL_CSV.with_suffix(".parquet")

print(f"Converting {FINAL_CSV} to {parquet_file} using PyArrow native streaming...")

# Configure Arrow CSV reader
# 1. Read options for optimal streaming chunk sizes
read_options = pv.ReadOptions(block_size=100 * 1024 * 1024) # 100MB chunks

# 2. Parse options
parse_options = pv.ParseOptions()

# 3. Read the CSV header to get all column names, then configure ConvertOptions
# to parse EVERY column as a string. This avoids CSV type inference mismatches (e.g., '45/46' in an integer column).
preview_reader = pv.open_csv(FINAL_CSV, read_options=pv.ReadOptions(block_size=1024*1024))
schema = preview_reader.schema
preview_reader.close()

column_types = {name: pa.string() for name in schema.names}
convert_options = pv.ConvertOptions(column_types=column_types)

# Open the CSV with the forced string conversion schema
reader = pv.open_csv(FINAL_CSV, read_options=read_options, parse_options=parse_options, convert_options=convert_options)

writer = None
try:
    with tqdm(desc="Converting to Parquet") as pbar:
        for record_batch in reader:
            if writer is None:
                # Set compression to snappy for fast and storage-efficient lookups
                writer = pq.ParquetWriter(parquet_file, record_batch.schema, compression='snappy')

            writer.write_batch(record_batch)
            pbar.update(record_batch.num_rows)
finally:
    if writer:
        writer.close()
    reader.close()

print(f"\nConversion complete! Parquet file saved at: {parquet_file}")
print(f"Parquet file size: {parquet_file.stat().st_size/1e9:.2f} GB")

Converting /content/nyc311_20200101_20261127_all_raw/nyc311_20200101_20261127_all_raw.csv to /content/nyc311_20200101_20261127_all_raw/nyc311_20200101_20261127_all_raw.parquet using PyArrow native streaming...


Converting to Parquet: 0it [00:00, ?it/s]


Conversion complete! Parquet file saved at: /content/nyc311_20200101_20261127_all_raw/nyc311_20200101_20261127_all_raw.parquet
Parquet file size: 2.51 GB


## 8. Optional: process all raw records in manageable batches

This does **not** change the downloaded records. It is only an example of how to analyze the large CSV without trying to hold every observation in RAM.

In [ ]:
# Optional example: count complaint types across the ENTIRE dataset.
# Reads one column at a time and retains the full source CSV on disk.
#
# from collections import Counter
# counts = Counter()
# files = [FINAL_CSV] if FINAL_CSV.exists() else [PARTS_DIR / p["file"] for p in saved]
# for filename in files:
#     for batch in pd.read_csv(filename, usecols=["complaint_type"], chunksize=200_000):
#         counts.update(batch["complaint_type"].fillna("Missing").value_counts().to_dict())
# print(counts.most_common(20))


## Notes / limitations

- **Each observation is preserved.** No daily grouping, sampling, deduplication, or column removal is applied to the raw downloads.
- The window criteria use `created_date`. Requests without a valid creation timestamp cannot be assigned to the selected date range.
- NYC updates the dataset daily and can revise existing records, so the separate requests are not a single atomic snapshot. Re-running with a newer end date in a new folder produces a new extraction.
- `MERGE_PARTS=True` duplicates storage until the part files are removed; the part files are intentionally kept for reliable resume.
- Sources: [NYC 311 dataset](https://data.cityofnewyork.us/d/erm2-nwe9); [Socrata row-limit guidance](https://support.socrata.com/hc/en-us/articles/202949268-How-to-query-more-than-1000-rows-of-a-dataset); [Socrata export and SODA3 docs](https://support.socrata.com/hc/en-us/articles/43491231777047-Using-the-SODA3-API).
